In [3]:
pip install polars


   ---------------------------------------- 0.0/865.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/865.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/865.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/865.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/865.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/865.8 kB ? eta -:--:--
   ------------ --------------------------- 262.1/865.8 kB ? eta -:--:--
   ------------ --------------------------- 262.1/865.8 kB ? eta -:--:--
   ------------ --------------------------- 262.1/865.8 kB ? eta -:--:--
   ------------ --------------------------- 262.1/865.8 kB ? eta -:--:--
   ------------ --------------------------- 262.1/865.8 kB ? eta -:--:--
   ----------------------- -------------- 524.3/865.8 kB 256.9 kB/s eta 0:00:02
   ----------------------- -------------- 524.3/865.8 kB 256.9 kB/s eta 0:00:02
   ----------------------- -------------- 524.3/8


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
pip install lightgbm

  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
pip install catboost

  Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl.metadata (1.5 kB)
Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl (101.7 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from lightgbm import LGBMClassifier, early_stopping
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

PROJECT_DIR = Path(r"D:\Y3S2\FDM\assignment\New folder")
DATA_PATH = PROJECT_DIR / "data" / "aeolus_slim.parquet"

# True: check the workflow with smaller samples; skip TEST.
# False: final training and TEST evaluation.
QUICK = True

OUTPUT_DIR = PROJECT_DIR / (
    "outputs_quick" if QUICK else "outputs_4class"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
CLASS_NAMES = ["On-time", "Minor", "Moderate", "Severe"]
LABELS = [0, 1, 2, 3]

TRAIN_END = pd.Timestamp("2024-08-31")
VAL_END = pd.Timestamp("2024-10-31")

TRAIN_LIMIT = 100_000 if QUICK else 300_000
VAL_LIMIT = 30_000 if QUICK else 200_000
RF_LIMIT = 30_000 if QUICK else 100_000

LOAD_COLS = [
    "FL_DATE", "OP_CARRIER", "ORIGIN", "DEST",
    "CRS_DEP_TIME", "CRS_ARR_TIME", "CRS_ELAPSED_TIME",
    "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK",
    "O_TEMP", "O_PRCP", "O_WSPD",
    "D_TEMP", "D_PRCP", "D_WSPD",
    "O_LATITUDE", "O_LONGITUDE",
    "D_LATITUDE", "D_LONGITUDE", "ARR_DELAY"
]

print("Data exists:", DATA_PATH.exists())
print("Mode:", "QUICK CHECK" if QUICK else "FINAL TRAINING")

Data exists: True
Mode: QUICK CHECK


In [24]:
CSV_PATH = PROJECT_DIR / "data" / "flight_with_weather_2024.csv"

df = pd.read_csv(
    CSV_PATH,
    usecols=LOAD_COLS,
    low_memory=False
)

df["FL_DATE"] = pd.to_datetime(
    df["FL_DATE"], errors="coerce"
).dt.normalize()

numeric_cols = [
    col for col in LOAD_COLS
    if col not in ["FL_DATE", "OP_CARRIER", "ORIGIN", "DEST"]
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

original_rows = len(df)

df = df.dropna(subset=["FL_DATE", "ARR_DELAY"]).copy()

df = df[
    df["FL_DATE"].between("2024-01-01", "2024-12-31")
].copy()

print("Original rows:", f"{original_rows:,}")
print("Usable 2024 rows:", f"{len(df):,}")
print("Earliest date:", df["FL_DATE"].min())
print("Latest date:", df["FL_DATE"].max())

print("\nRows per month:")
print(
    df["FL_DATE"].dt.to_period("M")
    .value_counts()
    .sort_index()
)

Original rows: 6,284,841
Usable 2024 rows: 6,284,841
Earliest date: 2024-01-01 00:00:00
Latest date: 2024-12-31 00:00:00

Rows per month:
FL_DATE
2024-01    517544
2024-02    507200
2024-04    568475
2024-05    590760
2024-06    591680
2024-07    603904
2024-08    596131
2024-09    571224
2024-10    600193
2024-11    563296
2024-12    574434
Freq: M, Name: count, dtype: int64


In [25]:
CAT_COLS = ["OP_CARRIER", "ORIGIN", "DEST"]


def make_features(data):
    """Use this same function when predicting new flights."""
    X = data.drop(columns=["FL_DATE", "ARR_DELAY"], errors="ignore").copy()

    dates = pd.to_datetime(data["FL_DATE"], errors="coerce")

    # Derive calendar features consistently from FL_DATE.
    X = X.drop(
        columns=["MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK"],
        errors="ignore"
    )

    X["month"] = dates.dt.month
    X["day_of_month"] = dates.dt.day
    X["day_of_week"] = dates.dt.dayofweek

    X["month_sin"] = np.sin(2 * np.pi * X["month"] / 12)
    X["month_cos"] = np.cos(2 * np.pi * X["month"] / 12)

    # Scheduled HHMM times. Treat 2400 as midnight.
    for col in ["CRS_DEP_TIME", "CRS_ARR_TIME"]:
        values = pd.to_numeric(X[col], errors="coerce")

        hours = np.floor(values / 100)
        minutes = values % 100

        valid = (
            (values == np.floor(values))
            & (
                (
                    hours.between(0, 23)
                    & minutes.between(0, 59)
                )
                | (values == 2400)
            )
        )

        hour = (hours % 24).where(valid)
        minute = minutes.where(valid)

        X[f"{col}_hour"] = hour
        X[f"{col}_minute"] = minute

        minutes_of_day = hour * 60 + minute
        X[f"{col}_sin"] = np.sin(
            2 * np.pi * minutes_of_day / 1440
        )
        X[f"{col}_cos"] = np.cos(
            2 * np.pi * minutes_of_day / 1440
        )

    X = X.drop(columns=["CRS_DEP_TIME", "CRS_ARR_TIME"])

    # Great-circle distance.
    coord_cols = [
        "O_LATITUDE", "O_LONGITUDE",
        "D_LATITUDE", "D_LONGITUDE"
    ]

    lat1, lon1, lat2, lon2 = [
        np.radians(pd.to_numeric(X[col], errors="coerce"))
        for col in coord_cols
    ]

    h = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin((lon2 - lon1) / 2) ** 2
    )

    X["distance_km"] = (
        6371 * 2 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))
    )

    X = X.drop(columns=coord_cols)

    for col in CAT_COLS:
        X[col] = (
            X[col].astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    numerical = X.columns.difference(CAT_COLS)
    X[numerical] = X[numerical].replace(
        [np.inf, -np.inf], np.nan
    )

    return X


def sample_rows(data, limit):
    if limit is None or len(data) <= limit:
        return data.copy()

    return data.sample(
        n=limit, random_state=SEED
    ).sort_values("FL_DATE").copy()


train_df = df[df["FL_DATE"] <= TRAIN_END]
val_df = df[
    (df["FL_DATE"] > TRAIN_END)
    & (df["FL_DATE"] <= VAL_END)
]
test_df = df[df["FL_DATE"] > VAL_END]

for name, part in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    if part.empty:
        raise ValueError(f"{name} is empty. Check your dataset dates.")

    print(
        f"{name}: {len(part):,} rows | "
        f"{part['FL_DATE'].min().date()} to "
        f"{part['FL_DATE'].max().date()}"
    )

train_df = sample_rows(train_df, TRAIN_LIMIT)
val_df = sample_rows(val_df, VAL_LIMIT)

# Keep the full test period for the final evaluation.
X_train_raw = make_features(train_df)
X_val_raw = make_features(val_df)


def make_labels(data):
    return np.digitize(
        data["ARR_DELAY"].to_numpy(),
        bins=[15, 45, 120]
    ).astype(np.int64)


y_train = make_labels(train_df)
y_val = make_labels(val_df)

for name, target in [
    ("Training", y_train),
    ("Validation", y_val)
]:
    counts = np.bincount(target, minlength=4)

    print(f"\n{name} class counts:")
    print(pd.Series(counts, index=CLASS_NAMES))

    if (counts == 0).any():
        raise ValueError(f"{name} is missing a target class.")

Training: 3,975,694 rows | 2024-01-01 to 2024-08-31
Validation: 1,171,417 rows | 2024-09-01 to 2024-10-31
Test: 1,137,730 rows | 2024-11-01 to 2024-12-31

Training class counts:
On-time     76412
Minor       12091
Moderate     7936
Severe       3561
dtype: int64

Validation class counts:
On-time     25643
Minor        2624
Moderate     1291
Severe        442
dtype: int64


In [26]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=np.float32
)

encoder.fit(X_train_raw[CAT_COLS])

FEATURE_COLS = X_train_raw.columns.tolist()


def encode_features(raw):
    X = raw.reindex(columns=FEATURE_COLS).copy()
    X[CAT_COLS] = encoder.transform(X[CAT_COLS])
    return X.astype(np.float32)


X_train = encode_features(X_train_raw)
X_val = encode_features(X_val_raw)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Feature count:", len(FEATURE_COLS))

Training shape: (100000, 24)
Validation shape: (30000, 24)
Feature count: 24


In [27]:
models = {}
training_seconds = {}

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

N_TREES = 150 if QUICK else 2000
PATIENCE = 30 if QUICK else 100


def train_and_store(name, model, fit_kwargs, X=X_train, y=y_train):
    print(f"\nTraining {name}...", flush=True)

    started = time.time()
    model.fit(X, y, **fit_kwargs)

    models[name] = model
    training_seconds[name] = time.time() - started

    print(
        f"{name} completed in "
        f"{training_seconds[name]:.1f} seconds",
        flush=True
    )


lgb_model = LGBMClassifier(
    objective="multiclass",
    num_class=4,
    n_estimators=N_TREES,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=SEED,
    verbosity=-1
)

train_and_store(
    "LightGBM",
    lgb_model,
    {
        "sample_weight": sample_weights,
        "eval_set": [(X_val, y_val)],
        "callbacks": [
            early_stopping(PATIENCE, verbose=False)
        ]
    }
)


xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=N_TREES,
    learning_rate=0.08,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=PATIENCE,
    n_jobs=-1,
    random_state=SEED
)

train_and_store(
    "XGBoost",
    xgb_model,
    {
        "sample_weight": sample_weights,
        "eval_set": [(X_val, y_val)],
        "verbose": False
    }
)


cat_model = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=N_TREES,
    learning_rate=0.08,
    depth=6,
    early_stopping_rounds=PATIENCE,
    random_seed=SEED,
    thread_count=-1,
    verbose=False,
    allow_writing_files=False
)

train_and_store(
    "CatBoost",
    cat_model,
    {
        "sample_weight": sample_weights,
        "eval_set": (X_val, y_val)
    }
)


rng = np.random.default_rng(SEED)

rf_indices = rng.choice(
    len(X_train),
    size=min(RF_LIMIT, len(X_train)),
    replace=False
)

if len(np.unique(y_train[rf_indices])) != 4:
    raise ValueError("RandomForest sample is missing a class.")

rf_model = RandomForestClassifier(
    n_estimators=60 if QUICK else 300,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=SEED
)

train_and_store(
    "RandomForest",
    rf_model,
    {},
    X=X_train.iloc[rf_indices].fillna(-999),
    y=y_train[rf_indices]
)


Training LightGBM...
LightGBM completed in 5.2 seconds

Training XGBoost...
XGBoost completed in 5.4 seconds

Training CatBoost...
CatBoost completed in 5.6 seconds

Training RandomForest...
RandomForest completed in 0.6 seconds


In [28]:
def get_proba(name, model, X):
    if name == "RandomForest":
        X = X.fillna(-999)

    proba = model.predict_proba(X)

    if not np.array_equal(model.classes_, LABELS):
        raise ValueError(
            f"{name}: unexpected class order {model.classes_}"
        )

    return proba


def macro_f1_fast(y, predictions):
    cm = np.bincount(
        y * 4 + predictions,
        minlength=16
    ).reshape(4, 4)

    tp = np.diag(cm)
    denominator = cm.sum(axis=0) + cm.sum(axis=1)

    scores = np.divide(
        2 * tp,
        denominator,
        out=np.zeros(4, dtype=float),
        where=denominator != 0
    )

    return scores.mean()


def tune_multipliers(proba, y):
    weights = np.ones(4)

    best_score = macro_f1_fast(
        y, proba.argmax(axis=1)
    )

    grid = np.arange(0.5, 6.01, 0.5)

    for _ in range(2):
        for class_id in [1, 2, 3]:
            for candidate in grid:
                trial = weights.copy()
                trial[class_id] = candidate

                predictions = (proba * trial).argmax(axis=1)
                score = macro_f1_fast(y, predictions)

                if score > best_score:
                    best_score = score
                    weights = trial

    return weights, best_score


comparison_rows = []
model_multipliers = {}

for name, model in models.items():
    print(f"Evaluating validation: {name}", flush=True)

    probabilities = get_proba(name, model, X_val)

    normal_predictions = probabilities.argmax(axis=1)

    multipliers, tuned_f1 = tune_multipliers(
        probabilities, y_val
    )

    tuned_predictions = (
        probabilities * multipliers
    ).argmax(axis=1)

    model_multipliers[name] = multipliers

    comparison_rows.append({
        "model": name,
        "train_seconds": training_seconds[name],
        "val_accuracy": accuracy_score(
            y_val, normal_predictions
        ),
        "val_macro_f1": f1_score(
            y_val,
            normal_predictions,
            labels=LABELS,
            average="macro",
            zero_division=0
        ),
        "val_macro_f1_tuned": tuned_f1,
        "val_accuracy_tuned": accuracy_score(
            y_val, tuned_predictions
        )
    })

comparison = pd.DataFrame(comparison_rows).sort_values(
    "val_macro_f1_tuned",
    ascending=False
).reset_index(drop=True)

comparison.to_csv(
    OUTPUT_DIR / "model_comparison_validation.csv",
    index=False
)

display(comparison.round(4))

best_name = comparison.loc[0, "model"]
best_model = models[best_name]
best_multipliers = model_multipliers[best_name]

print("\nSelected model:", best_name)
print("Probability multipliers:", best_multipliers)

Evaluating validation: LightGBM
Evaluating validation: XGBoost
Evaluating validation: CatBoost
Evaluating validation: RandomForest


,model,train_seconds,val_accuracy,val_macro_f1,val_macro_f1_tuned,val_accuracy_tuned
0,RandomForest,0.6407,0.6749,0.2660,0.2660,0.6749
1,XGBoost,5.3702,0.6235,0.2632,0.2652,0.6522
2,LightGBM,5.1826,0.6196,0.2639,0.2639,0.6196
3,CatBoost,5.6397,0.6050,0.2545,0.2557,0.6289



Selected model: RandomForest
Probability multipliers: [1. 1. 1. 1.]
